# Volume 11, Chapter 10: Random matrices I, the semicircle (Python companion)

Exercises first, solutions at the end. Universality of the semicircle and where it fails, Stieltjes transforms and their inversion, the Marchenko–Pastur law and the Narayana polynomials, the largest eigenvalue of the GOE and its $n^{-2/3}$ fluctuations, the spiked Wigner matrix of Theorem 10.11, and a statistics application: **when principal component analysis sees a signal, and when it sees only noise** (the BBP transition). Requires `numpy` and `scipy`.

In [1]:
import math
import numpy as np
from scipy import stats
rng = np.random.default_rng(1110)

def wigner(n, sampler):
    X = sampler((n, n)); X = np.triu(X, 1); X = X + X.T
    return X / math.sqrt(n)

def goe(n):
    G = rng.standard_normal((n, n)); return (G + G.T) / math.sqrt(2 * n)   # off-diagonal N(0,1/n), diagonal N(0,2/n)

semicircle = lambda x: np.sqrt(np.clip(4 - x**2, 0, None)) / (2 * np.pi)

## Exercises

**P10.1 (★) Universality and its limits.** For $n=2000$, compare the eigenvalue histogram of $X/\sqrt n$ with the semicircle for entries that are random signs, Gaussian, Student $t$ with $3$ degrees of freedom (finite variance, infinite fourth moment) normalised to variance $1$, and Cauchy (infinite variance, scaled by $\sqrt n\log n$ instead). Look at the extreme eigenvalues.

**P10.2 (★★) Stieltjes transforms.** For a GOE matrix with $n=2000$, compute $s_n(z)=\frac1n\operatorname{tr}(W-z)^{-1}$ at $z=x+i\eta$ and compare with $s_{\mathrm{sc}}(z)$ of Proposition 10.5. Recover the density by $\frac1\pi\operatorname{Im}s_n(x+i\eta)$ for $\eta=0.3,0.05,0.002$: what goes wrong when $\eta$ is of order $\frac1n$?

**P10.3 (★) Marchenko–Pastur.** For $N=4000$ and $\gamma=0.1,0.5,0.9$, compare the eigenvalue histogram of $S=\frac1NXX^{\mathsf T}$ with $\mu_\gamma$, and its moments with the Narayana polynomials $m_k(\gamma)=\sum_rN(k,r)\gamma^{r-1}$.

**P10.4 (★★) The edge.** Estimate $\mathbb E\lambda_{\max}$ and the standard deviation of $\lambda_{\max}$ for the GOE with $n=100,200,\dots,1600$. Check that $2-\mathbb E\lambda_{\max}$ and the standard deviation decay like $n^{-2/3}$, and compare $n^{2/3}(\lambda_{\max}-2)$ with the Tracy–Widom law (mean $-1.2065$, standard deviation $1.2680$ for the GOE).

**P10.5 (★★) The spiked Wigner matrix.** For $n=1500$ and $\theta$ from $0$ to $3$, compute the largest eigenvalue of $W+\theta vv^{\mathsf T}$ and the overlap $\langle x,v\rangle^2$ of its eigenvector with $v$, and compare with Theorem 10.11 and Exercise 10.8.

**P10.6 (★★★) Statistics application: when does PCA see the signal?** Generate $N=1000$ samples in dimension $p=500$ ($\gamma=\frac12$) from a Gaussian with covariance $I+\theta vv^{\mathsf T}$. For $\theta$ from $0$ to $3$, compute the top eigenvalue of the sample covariance and the overlap of the first principal component with $v$, and compare with the BBP predictions: edge $(1+\sqrt\gamma)^2$ and overlap $0$ below $\theta=\sqrt\gamma$, eigenvalue $(1+\theta)(1+\frac\gamma\theta)$ and overlap $\frac{1-\gamma/\theta^2}{1+\gamma/\theta}$ above.

## Solutions

### P10.1

**How to find it.** Same matrices, different entry laws; normalise each to variance $1$ ($t_3$ has variance $3$). Histograms are compared with the semicircle density at the bin centres through a total variation distance.

In [2]:
n = 2000
samplers = {
    "random signs": lambda s: rng.choice([-1.0, 1.0], size=s),
    "Gaussian": lambda s: rng.standard_normal(s),
    "Student t_3": lambda s: rng.standard_t(3, size=s) / math.sqrt(3),
}
edges = np.linspace(-2.5, 2.5, 51); mids = (edges[1:] + edges[:-1]) / 2; w = edges[1] - edges[0]
for name, f in samplers.items():
    ev = np.linalg.eigvalsh(wigner(n, f))
    h, _ = np.histogram(ev, bins=edges)
    tv = 0.5 * np.abs(h / n - semicircle(mids) * w).sum() + 0.5 * np.mean(np.abs(ev) > 2.5)
    print(f"{name:>13}: distance to semicircle {tv:.3f};  extreme eigenvalues {ev.min():.3f}, {ev.max():.3f}")
C = rng.standard_cauchy((n, n)); C = np.triu(C, 1); C = C + C.T
evc = np.linalg.eigvalsh(C / (math.sqrt(n) * math.log(n)))
print(f"       Cauchy: fraction of eigenvalues in [-2, 2]: {np.mean(np.abs(evc) <= 2):.3f}; largest |eigenvalue| {np.abs(evc).max():.1f}")

 random signs: distance to semicircle 0.010;  extreme eigenvalues -1.985, 1.982


     Gaussian: distance to semicircle 0.009;  extreme eigenvalues -1.993, 1.998


  Student t_3: distance to semicircle 0.017;  extreme eigenvalues -2.635, 2.613


       Cauchy: fraction of eigenvalues in [-2, 2]: 0.270; largest |eigenvalue| 3506.5


Signs and Gaussians give the semicircle (distance about $0.01$, the size of the histogram noise) with extreme eigenvalues close to $\pm2$. With $t_3$ entries the bulk is still a semicircle (the law only needs a finite variance, by a truncation argument), but the extreme eigenvalues are far outside $[-2,2]$: a few huge entries create eigenvalues of their own size, and Bai and Yin's theorem says that a finite fourth moment is exactly what prevents this. Cauchy entries have no variance at all, and there is no semicircle: the spectrum is heavy-tailed (Ben Arous and Guionnet, 2008, describe the limit).

### P10.2

**How to find it.** $s_n(z)=\frac1n\sum_i\frac1{\lambda_i-z}$ from the eigenvalues. Choose the branch of $\sqrt{z^2-4}$ with $\sqrt{z^2-4}\sim z$: $\sqrt{z-2}\sqrt{z+2}$ with principal square roots does it on $\mathbb C^+$.

In [3]:
ev = np.linalg.eigvalsh(goe(2000))
s_sc = lambda z: (-z + np.sqrt(z - 2) * np.sqrt(z + 2)) / 2
s_n = lambda z: np.mean(1 / (ev - z))
for z in (0.5 + 0.5j, 1.9 + 0.1j, 3 + 0.01j):
    print(f"z = {z}: s_n = {s_n(z):.4f}   s_sc = {s_sc(z):.4f}")
xs = np.array([-1.5, 0.0, 1.0, 1.9])
print("density at", xs, ":", np.round(semicircle(xs), 4))
for eta in (0.3, 0.05, 0.002):
    print(f"eta = {eta:>5}: (1/pi) Im s_n =", np.round([s_n(x + 1j * eta).imag / np.pi for x in xs], 4))

z = (0.5+0.5j): s_n = -0.1867+0.7518j   s_sc = -0.1876+0.7519j
z = (1.9+0.1j): s_n = -0.8160+0.2962j   s_sc = -0.8123+0.2949j
z = (3+0.01j): s_n = -0.3821+0.0017j   s_sc = -0.3820+0.0017j
density at [-1.5  0.   1.   1.9] : [0.2105 0.3183 0.2757 0.0994]
eta =   0.3: (1/pi) Im s_n = [0.1742 0.2734 0.2333 0.1   ]
eta =  0.05: (1/pi) Im s_n = [0.2016 0.3082 0.2674 0.0952]
eta = 0.002: (1/pi) Im s_n = [0.2487 0.3443 0.2774 0.1759]


The transform matches $s_{\mathrm{sc}}$ to two or three digits, including at $z=3+0.01i$ outside the spectrum, where no eigenvalue lies near $z$. The inversion recovers the density: $\eta=0.3$ smooths too much (it convolves with a Cauchy density of width $0.3$, which flattens the peak and fattens the edge), $\eta=0.05$ is accurate, and $\eta=0.002$, comparable with the spacing $\frac1{n\rho(x)}\approx0.0016$ of eigenvalues, makes $\operatorname{Im}s_n$ a sum of a few spiky Cauchy bumps that depends on where the nearest eigenvalues fall. The local semicircle law says the approximation holds down to $\eta\gg\frac1n$, and no further.

### P10.3

**How to find it.** The density of Theorem 10.8, and $N(k,r)=\frac1k\binom kr\binom k{r-1}$ (Exercise 10.7).

In [4]:
def mp_density(x, g):
    a, b = (1 - math.sqrt(g))**2, (1 + math.sqrt(g))**2
    return np.where((x > a) & (x < b), np.sqrt(np.clip((b - x) * (x - a), 0, None)) / (2 * np.pi * g * np.maximum(x, 1e-12)), 0)

def narayana_moment(k, g):
    return sum(math.comb(k, r) * math.comb(k, r - 1) // k * g**(r - 1) for r in range(1, k + 1))

N = 4000
for g in (0.1, 0.5, 0.9):
    p = int(g * N); X = rng.standard_normal((p, N)); ev = np.linalg.eigvalsh(X @ X.T / N)
    a, b = (1 - math.sqrt(g))**2, (1 + math.sqrt(g))**2
    edges = np.linspace(a, b, 21); mids = (edges[1:] + edges[:-1]) / 2
    h, _ = np.histogram(ev, bins=edges)
    print(f"gamma = {g}: support [{a:.3f}, {b:.3f}], eigenvalues in [{ev.min():.3f}, {ev.max():.3f}]")
    print("   histogram / density in 5 bins:", np.round((h / (p * (edges[1] - edges[0])) / mp_density(mids, g))[2::4], 3))
    print("   moments k = 1..5:", [round(float(np.mean(ev**k)), 3) for k in range(1, 6)])
    print("   Narayana        :", [round(narayana_moment(k, g), 3) for k in range(1, 6)])

gamma = 0.1: support [0.468, 1.732], eigenvalues in [0.480, 1.732]
   histogram / density in 5 bins: [1.003 0.958 1.068 0.974 0.977]
   moments k = 1..5: [1.0, 1.101, 1.311, 1.663, 2.214]
   Narayana        : [1.0, 1.1, 1.31, 1.661, 2.21]


gamma = 0.5: support [0.086, 2.914], eigenvalues in [0.087, 2.906]
   histogram / density in 5 bins: [0.991 1.003 0.963 1.015 1.007]
   moments k = 1..5: [1.001, 1.502, 2.756, 5.64, 12.354]
   Narayana        : [1.0, 1.5, 2.75, 5.625, 12.312]


gamma = 0.9: support [0.003, 3.797], eigenvalues in [0.003, 3.791]
   histogram / density in 5 bins: [0.997 0.996 1.002 0.969 1.018]
   moments k = 1..5: [1.0, 1.901, 4.512, 11.994, 34.155]
   Narayana        : [1.0, 1.9, 4.51, 11.989, 34.146]


The histograms follow $\mu_\gamma$, and the moments agree with the Narayana polynomials up to the finite-size corrections ($p$ is finite and the ratio is exactly $\gamma$ only in the limit). For $\gamma=0.9$ the support starts at $0.0026$: almost singular matrices, which is why least squares with nearly as many variables as samples is so unstable. The smallest eigenvalue sits at the edge $(1-\sqrt\gamma)^2$, as does the largest at $(1+\sqrt\gamma)^2$ (Exercise 10.5 for the upper edge).

### P10.4

**How to find it.** Many independent GOE matrices for each $n$; fit the slopes of $\log(2-\mathbb E\lambda_{\max})$ and $\log\operatorname{sd}$ against $\log n$.

In [5]:
ns = [100, 200, 400, 800, 1600]; means, sds = [], []
for n in ns:
    reps = max(40, 64000 // n)
    lm = np.array([np.linalg.eigvalsh(goe(n))[-1] for _ in range(reps)])
    means.append(lm.mean()); sds.append(lm.std())
    scaled = n**(2 / 3) * (lm - 2)
    print(f"n = {n:>4} ({reps:>3} samples): E lambda_max = {lm.mean():.4f}, sd = {lm.std():.4f};  n^(2/3)(lambda_max - 2): mean {scaled.mean():.3f}, sd {scaled.std():.3f}")
sl_m = np.polyfit(np.log(ns), np.log(2 - np.array(means)), 1)[0]
sl_s = np.polyfit(np.log(ns), np.log(sds), 1)[0]
print(f"fitted exponents: 2 - E lambda_max ~ n^{sl_m:.3f},  sd ~ n^{sl_s:.3f}   (prediction -2/3 = -0.667)")

n =  100 (640 samples): E lambda_max = 1.9375, sd = 0.0575;  n^(2/3)(lambda_max - 2): mean -1.346, sd 1.238


n =  200 (320 samples): E lambda_max = 1.9615, sd = 0.0369;  n^(2/3)(lambda_max - 2): mean -1.316, sd 1.262


n =  400 (160 samples): E lambda_max = 1.9784, sd = 0.0222;  n^(2/3)(lambda_max - 2): mean -1.175, sd 1.203


n =  800 ( 80 samples): E lambda_max = 1.9838, sd = 0.0162;  n^(2/3)(lambda_max - 2): mean -1.395, sd 1.399


n = 1600 ( 40 samples): E lambda_max = 1.9911, sd = 0.0090;  n^(2/3)(lambda_max - 2): mean -1.216, sd 1.234
fitted exponents: 2 - E lambda_max ~ n^-0.688,  sd ~ n^-0.653   (prediction -2/3 = -0.667)


$\mathbb E\lambda_{\max}$ is below $2$, as Theorem 10.10 guarantees, and both the gap and the standard deviation decay with exponents close to $-\frac23$, much faster than the $n^{-1/2}$ of Gaussian concentration. The rescaled fluctuations have mean and standard deviation close to those of the Tracy–Widom law for $\beta=1$ ($-1.21$ and $1.27$), within the statistical error of a few hundred samples. Concentration inequalities give the right centre but the wrong scale: the eigenvalues near the edge repel each other, and the largest one is pinned much more tightly than a generic Lipschitz function of $n^2$ Gaussians.

### P10.5

**How to find it.** By orthogonal invariance one may take $v=e_1$. The predictions: $\max(2,\theta+\frac1\theta)$ and overlap $\max(0,1-\theta^{-2})$.

In [6]:
n = 1500; W = goe(n)
print(" theta   lambda_max   predicted   overlap   predicted")
for theta in (0.0, 0.5, 0.9, 1.0, 1.1, 1.5, 2.0, 3.0):
    M = W.copy(); M[0, 0] += theta
    val, vec = np.linalg.eigh(M)
    pred = theta + 1 / theta if theta > 1 else 2.0
    print(f"{theta:>6.1f}   {val[-1]:>10.4f}   {pred:>9.4f}   {vec[0, -1]**2:>7.4f}   {max(0, 1 - theta**-2 if theta > 0 else 0):>9.4f}")

 theta   lambda_max   predicted   overlap   predicted


   0.0       1.9867      2.0000    0.0043      0.0000


   0.5       1.9906      2.0000    0.0141      0.0000


   0.9       2.0041      2.0000    0.0740      0.0000


   1.0       2.0139      2.0000    0.1262      0.0000


   1.1       2.0304      2.0091    0.2080      0.1736


   1.5       2.1885      2.1667    0.5530      0.5556


   2.0       2.5218      2.5000    0.7516      0.7500


   3.0       3.3575      3.3333    0.8913      0.8889


Below $\theta=1$ the top eigenvalue stays at the edge and its eigenvector ignores $v$ (overlap $0.004$ at $\theta=0$, $0.014$ at $\theta=\frac12$); above, the eigenvalue follows $\theta+\frac1\theta$ and the overlap follows $1-\theta^{-2}$. Near $\theta=1$ the finite-$n$ transition is smooth over a window of width about $n^{-1/3}\approx0.09$ (hence the overlap $0.07$ at $\theta=0.9$), where the outlier and the edge eigenvalues interact.

### P10.6

**How to find it.** Samples are $\Sigma^{1/2}Z$ with $Z$ standard Gaussian and $\Sigma=I+\theta vv^{\mathsf T}$; with $v=e_1$ this multiplies the first coordinate by $\sqrt{1+\theta}$. The first principal component is the top eigenvector of $S=\frac1N\sum x_ix_i^{\mathsf T}$.

In [7]:
p, N = 500, 1000; g = p / N
print(f"threshold sqrt(gamma) = {math.sqrt(g):.3f}, bulk edge (1 + sqrt(gamma))^2 = {(1 + math.sqrt(g))**2:.3f}")
print(" theta   top eigenvalue   predicted   overlap   predicted")
for theta in (0.0, 0.3, 0.6, 0.8, 1.0, 1.5, 2.0, 3.0):
    Z = rng.standard_normal((p, N)); Z[0] *= math.sqrt(1 + theta)
    val, vec = np.linalg.eigh(Z @ Z.T / N)
    if theta > math.sqrt(g):
        pe, po = (1 + theta) * (1 + g / theta), (1 - g / theta**2) / (1 + g / theta)
    else:
        pe, po = (1 + math.sqrt(g))**2, 0.0
    print(f"{theta:>6.1f}   {val[-1]:>14.4f}   {pe:>9.4f}   {vec[0, -1]**2:>7.4f}   {po:>9.4f}")

threshold sqrt(gamma) = 0.707, bulk edge (1 + sqrt(gamma))^2 = 2.914
 theta   top eigenvalue   predicted   overlap   predicted
   0.0           2.8879      2.9142    0.0000      0.0000
   0.3           2.8861      2.9142    0.0034      0.0000
   0.6           2.8659      2.9142    0.0095      0.0000
   0.8           2.9335      2.9250    0.0735      0.1346
   1.0           2.9552      3.0000    0.2981      0.3333
   1.5           3.5350      3.3333    0.6180      0.5833


   2.0           3.8122      3.7500    0.7281      0.7000
   3.0           4.7230      4.6667    0.8113      0.8095


With $500$ variables and $1000$ samples, a direction whose variance is $1.6$ times the others ($\theta=0.6$) is invisible: the top sample eigenvalue sits at the edge of the noise, and the first principal component has essentially nothing to do with $v$. Doubling the variance ($\theta=1$) makes the signal appear, but the estimated direction is still poor (overlap about $0.33$), and the top eigenvalue overestimates the true variance $1+\theta$. In practice: before trusting a principal component in high dimension, compare its eigenvalue with the Marchenko–Pastur edge $(1+\sqrt{p/N})^2$ times the noise level; components below it carry no information (this is the basis of "parallel analysis" and of eigenvalue shrinkage estimators, Donoho, Gavish and Johnstone, *Annals of Statistics* 46, 2018).